<a href="https://colab.research.google.com/github/tetigai/gazo/blob/main/1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!apt-get update
!apt-get install -y libopencv-dev

Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Hit:3 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:4 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:5 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:6 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:7 https://cli.github.com/packages stable/main amd64 Packages [354 B]
Get:8 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [89.0 kB]
Get:9 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:10 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:11 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [2,970 kB]
Hit:12 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:13 http://archive.ubuntu.com/ubuntu jammy-updates/restricted amd64 Packages [7,240 kB]
Get:14 https

In [2]:
%%writefile temp.cpp
#include <opencv2/opencv.hpp>
#include <chrono>
#include <iostream>

using namespace cv;
using namespace std;
using namespace std::chrono;

/* 二重ループでネガポジ変換 */
void negative_loop(const Mat& src, Mat& dst)
{
    dst.create(src.size(), src.type());

    int bytes_per_row = src.cols * src.channels();

    for (int y = 0; y < src.rows; y++) {
        const uchar* src_row = src.ptr<uchar>(y);
        uchar* dst_row = dst.ptr<uchar>(y);

        for (int x = 0; x < bytes_per_row; x++) {
            dst_row[x] = 255 - src_row[x];
        }
    }
}

int main()
{
    Mat gray_src = imread("girl.png", IMREAD_GRAYSCALE);
    Mat color_src = imread("lenna.png", IMREAD_COLOR);

    if (gray_src.empty()) {
        cerr << "girl.png" << " を読み込めませんでした．" << endl;
        return 1;
    }

    if (color_src.empty()) {
        cerr << "lenna.png" << " を読み込めませんでした．" << endl;
        return 1;
    }

    Mat gray_loop_dst, gray_bitwise_dst;
    Mat color_loop_dst, color_bitwise_dst;

    auto start1 = steady_clock::now();
    negative_loop(gray_src, gray_loop_dst);
    auto end1 = steady_clock::now();

    auto start2 = steady_clock::now();
    bitwise_not(gray_src, gray_bitwise_dst);
    auto end2 = steady_clock::now();

    auto start3 = steady_clock::now();
    negative_loop(color_src, color_loop_dst);
    auto end3 = steady_clock::now();

    auto start4 = steady_clock::now();
    bitwise_not(color_src, color_bitwise_dst);
    auto end4 = steady_clock::now();

    double gray_loop_time = duration<double, milli>(end1 - start1).count();
    double gray_bitwise_time = duration<double, milli>(end2 - start2).count();
    double color_loop_time = duration<double, milli>(end3 - start3).count();
    double color_bitwise_time = duration<double, milli>(end4 - start4).count();

    imwrite("gray_negative_loop.png", gray_loop_dst);
    imwrite("gray_negative_bitwise.png", gray_bitwise_dst);
    imwrite("color_negative_loop.png", color_loop_dst);
    imwrite("color_negative_bitwise.png", color_bitwise_dst);

    cout << "[モノクロ画像]" << endl;
    cout << "二重ループ版   : " << gray_loop_time << " ms" << endl;
    cout << "bitwise_not版 : " << gray_bitwise_time << " ms" << endl;

    cout << endl;

    cout << "[カラー画像]" << endl;
    cout << "二重ループ版   : " << color_loop_time << " ms" << endl;
    cout << "bitwise_not版 : " << color_bitwise_time << " ms" << endl;

    return 0;
}

Writing temp.cpp


In [3]:
!g++ temp.cpp -o temp `pkg-config --cflags --libs opencv4`

In [29]:
!./temp

[モノクロ画像]
二重ループ版   : 0.254359 ms
bitwise_not版 : 0.078064 ms

[カラー画像]
二重ループ版   : 0.792431 ms
bitwise_not版 : 0.146042 ms
